# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, with each dataset field referenced by its Croissant schema `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. The structure of a Croissant schema includes one or more record sets, fields, and columns. We will print all available record sets along with their fields and columns, always referencing entities by their `@id`.

In [ ]:
# Helper: walk the Croissant schema to find all record sets and their fields by @id

def get_record_sets(ds):
    recs = []
    if hasattr(ds, 'metadata') and hasattr(ds.metadata, 'record_sets'):
        record_sets = ds.metadata.record_sets
    elif hasattr(ds.metadata, 'recordSet'):
        record_sets = ds.metadata.recordSet
    else:
        # Older schema spelling fallback
        record_sets = getattr(ds.metadata, 'record_sets', [])
    return record_sets

record_sets = get_record_sets(dataset)
print('Available Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    # List columns/fields
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for fld in fields:
        print(f"    - @id: {fld['@id']} | name: {fld.get('name', 'N/A')}")
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for col in columns:
            print(f"    - @id: {col['@id']} | name: {col.get('name', 'N/A')}")
    print()
if not record_sets:
    print(' (No explicit recordSet definitions found. Attempting to enumerate implicit CSV/TSV records in fileObjects as fallback.)')
    if hasattr(dataset.metadata, 'distribution'):
        for dist in dataset.metadata.distribution:
            print(f"- Distribution @id: {dist['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above. If there are multiple record sets, you can loop over them. Otherwise, target the principal record set.

In [ ]:
# Attempt to find the main record set from overview above
# If no recordSet in schema, we may need to guess based on distribution/FileObject

main_record_set_id = None
record_set_ids = []
record_sets = get_record_sets(dataset)
if record_sets:
    for rs in record_sets:
        record_set_ids.append(rs['@id'])
    # Use the first by default
    if record_set_ids:
        main_record_set_id = record_set_ids[0]
else:
    # Fallback for schemas defined only by distribution
    if hasattr(dataset.metadata, 'distribution') and dataset.metadata.distribution:
        dist_ids = [dist['@id'] for dist in dataset.metadata.distribution]
        main_record_set_id = dist_ids[0]
        record_set_ids = dist_ids
    else:
        raise ValueError("No record sets or distribution found in the metadata.")

print(f"Record Set IDs: {record_set_ids}")

dataframes = {}
for record_set in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded DataFrame for record set: {record_set} with shape {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Failed to load records for record_set {record_set} due to: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on numeric field, normalization, and grouping. 
### Reference all fields by their `@id`. 
If you don't know which field is numeric, check the field names above and select one for demonstration.

In [ ]:
# --- Demo: Find numeric field for EDA ---
df = dataframes[main_record_set_id]
numeric_candidate = None
for col in df.columns:
    # Try to infer numeric columns
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_candidate = col
        print(f"Numeric field candidate by @id: '{col}'")
        break
if not numeric_candidate:
    # Try to parse numeric
    for col in df.columns:
        try:
            _ = pd.to_numeric(df[col])
            numeric_candidate = col
            print(f"Numeric field candidate by @id (after conversion): '{col}'")
            df[col] = pd.to_numeric(df[col], errors='coerce')
            break
        except:
            continue

# Set for the next block
numeric_field_id = numeric_candidate if numeric_candidate else df.columns[0]  # fallback to first column

print(f"Using numeric field @id: {numeric_field_id}")

In [ ]:
# Filter, normalize, and group data using the chosen numeric_field_id
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'iufc' else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (by @id):")
display(filtered_df.head())

# Normalize this numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a possible group field that is not numeric
group_field = None
for col in df.columns:
    if col != numeric_field_id and df[col].dtype == 'object':
        group_field = col
        break
print(f"Grouping by @id: {group_field if group_field else '(none found)'}")

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field} (both by @id):")
    display(grouped_df.head())

## 5. Visualization
Visualize field distributions and relationships. All axes/labels should reference fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If a group field was found, boxplot
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
We have loaded and explored the FAIR<sup>2</sup> dataset package: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. 
- Dataset fields were referenced by their unique `@id`.
- Numeric fields were filtered and normalized; record grouping and visualization was demonstrated.
- For further work, use field `@id`s from the record set overview in additional analysis or ML workflows.